# 12: ROC Curves, Precision-Recall Curves, and Clustering Evaluation

**Author:** Dr Rob Lyon 

**Version:** 1.0

## Code & License
Copyright © 2026 Robert Lyon. All Rights Reserved.

This notebook and all associated materials are the intellectual property of the author.

Permission is granted solely to read, study, and analyse this material for personal educational purposes. No other rights are granted.

Without the prior written consent of the author, you may not:

* Copy, reproduce, redistribute, publish, transmit, or display this work in whole or in part.
* Modify, adapt, transform, translate, or create derivative works based on this material.
* Incorporate any portion of this work into another project, publication, product, model, dataset, or codebase.
* Use this material for commercial purposes.
* Remove or alter this copyright notice.

All intellectual property rights, including copyright and any derivative rights, remain exclusively vested in the author.

Access to this material does not constitute a transfer of ownership, license, or any other intellectual property rights except as expressly stated above.

---

**Note:**

This notebook is designed to be viewed from within JupyterLab. The Table of Contents and "Back to Table of Contents" links rely on JupyterLab's anchor handling to jump between sections.

If you open this notebook in another tool, such as PyCharm or Visual Studio Code, these anchor links may not work as expected. You can still navigate the notebook in those tools by using the headings directly, most editors provide an outline or contents panel that lists them for you.

---
## Table of Contents

1. [Introduction](#1.-Introduction)
2. [Useful Links](#2.-Useful-Links)
3. [Libraries and Environment Setup](#3.-Libraries-and-Environment-Setup)
4. [From Hard Predictions to Scores](#4.-From-Hard-Predictions-to-Scores)
    - 4.1 [A Running Example: Breast Cancer Diagnosis](#4.1-A-Running-Example:-Breast-Cancer-Diagnosis)
    - 4.2 [Building the ROC Curve](#4.2-Building-the-ROC-Curve)
    - 4.3 [Generating ROC Curves for Other Classifiers](#4.3-Generating-ROC-Curves-for-Other-Classifiers)
    - 4.4 [The Area Under the Curve (AUC)](#4.4-The-Area-Under-the-Curve-(AUC))
5. [Reading the ROC Curve](#5.-Reading-the-ROC-Curve)
    - 5.1 [Comparing Multiple Classifiers](#5.1-Comparing-Multiple-Classifiers)
6. [Multi-Class ROC Curves](#6.-Multi-Class-ROC-Curves)
    - 6.1 [One-vs-Rest (OvR)](#6.1-One-vs-Rest-(OvR))
    - 6.2 [One-vs-One (OvO)](#6.2-One-vs-One-(OvO))
    - 6.3 [A Key Limitation of OvR](#6.3-A-Key-Limitation-of-OvR)
7. [Weaknesses of ROC Curves](#7.-Weaknesses-of-ROC-Curves)
8. [Precision-Recall Curves](#8.-Precision-Recall-Curves)
    - 8.1 [Computing PR Curves in Scikit-Learn](#8.1-Computing-PR-Curves-in-Scikit-Learn)
    - 8.2 [Multi-Class PR Curves](#8.2-Multi-Class-PR-Curves)
9. [Evaluating Unsupervised Methods](#9.-Evaluating-Unsupervised-Methods)
    - 9.1 [The Core Problem](#9.1-The-Core-Problem)
    - 9.2 [WCSS and the Elbow Method](#9.2-WCSS-and-the-Elbow-Method)
    - 9.3 [Silhouette Score](#9.3-Silhouette-Score)
    - 9.4 [Davies-Bouldin Index and Dunn Index](#9.4-Davies-Bouldin-Index-and-Dunn-Index)
10. [Summary](#10.-Summary)
11. [References](#11.-References)

---

## 1. Introduction

Welcome to **Notebook 12**, the final one in this series. Notebooks 10 and 11 built a solid foundation in classifier evaluation, covering cross-validation, the confusion matrix, accuracy, precision, recall, specificity, and the challenges of class imbalance. This notebook completes the evaluation toolkit with more advanced techniques, extending your evaluation repertoire well beyond simple metrics. The first half focuses on the **ROC curve** in depth: what it measures, how to read its shape, how to compare classifiers, how to extend it to multi-class problems, and, critically, its real limitations on imbalanced data. We then introduce the **Precision-Recall curve** as the preferred alternative when classes are unequal. The second half turns to **evaluating unsupervised methods**: how do you judge cluster quality when there are no labels?

### Learning Objectives

By the end of this notebook you should be able to:

- Explain what a decision score (soft prediction) is and how varying the decision threshold traces a path through TPR-FPR space.
- Construct and interpret an ROC curve, and calculate AUC both manually and using scikit-learn.
- Describe the probabilistic interpretation of AUC.
- Recognise the five common ROC curve shapes (perfect, good, random, poor, misleading) and explain what each implies about a classifier.
- Compare multiple classifiers using ROC curves and explain why crossing curves complicate the comparison.
- Extend ROC evaluation to multi-class problems using the One-vs-Rest strategy.
- Describe the key weaknesses of ROC curves and AUC, particularly on imbalanced datasets.
- Construct and interpret a Precision-Recall curve and Average Precision score, and know when to prefer it over ROC.
- Define compactness and separation as the two goals of clustering evaluation.
- Compute WCSS, silhouette score, Davies-Bouldin Index, and Dunn Index using scikit-learn.
- Use multiple clustering metrics together to select an appropriate value of $k$.


---




## 2. Useful Links

| Link | Description |
|------|-------------|
| [scikit-learn: ROC metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#roc-metrics) | Official documentation for `roc_curve`, `roc_auc_score`, and related functions. The worked examples are particularly useful for understanding the API. |
| [scikit-learn: Precision-Recall](https://scikit-learn.org/stable/modules/model_evaluation.html#precision-recall-f-measure-metrics) | Documentation for `precision_recall_curve` and `average_precision_score`, including guidance on interpreting Average Precision. |
| [scikit-learn: Clustering evaluation](https://scikit-learn.org/stable/modules/clustering.html#clustering-performance-evaluation) | Complete reference for all clustering metrics available in scikit-learn, including silhouette score and Davies-Bouldin Index. |
| [Wikipedia: Receiver operating characteristic](https://en.wikipedia.org/wiki/Receiver_operating_characteristic) | A thorough treatment of ROC curves, AUC, and the probabilistic interpretation; useful for self-study and exam preparation. |


---

## 3. Libraries and Environment Setup

### 3.1 Working Environment

To run the code in this notebook, you will need a Python environment with the required libraries listed below installed.

The easiest way to get started is to use the **Project IO** Docker container, which provides a fully configured and tested environment containing all required Python packages and supporting dependencies. Instructions for downloading and running the container can be found in the **project-io** GitHub repository:

https://github.com/scienceguyrob/project-io

Using the Docker container ensures that the notebooks run in exactly the same environment in which they were developed and tested.

If you prefer not to use Docker, a good alternative is to install the [Anaconda Distribution](https://www.anaconda.com/products/distribution). You can then create a new Python environment and install the required libraries using either `conda install` or `pip install`.
### 3.2 Libraries Used in This Notebook

All sections of this notebook require the external libraries listed below.

| Library | Purpose | Documentation |
|---------|---------|---------------|
| **NumPy** (`numpy`) | Fast numerical arrays, mathematical functions, and random number generation. | [numpy.org](https://numpy.org/doc/stable/) |
| **Pandas** (`pandas`) | Tabular data structures used for organising score outputs and displaying results. | [pandas.pydata.org](https://pandas.pydata.org/docs/) |
| **Matplotlib** (`matplotlib.pyplot`) | Core plotting library. Used for all figures in this notebook. | [matplotlib.org](https://matplotlib.org/stable/) |
| **Seaborn** (`seaborn`) | Statistical visualisation library built on Matplotlib. Used to set a consistent plot theme. | [seaborn.pydata.org](https://seaborn.pydata.org/) |
| **scikit-learn** (`sklearn`) | Provides all classifiers, clustering algorithms, datasets, preprocessing utilities, and evaluation metrics used in this notebook. | [scikit-learn.org](https://scikit-learn.org/stable/) |
| **SciPy** (`scipy.spatial.distance`) | Used for computing pairwise distance matrices in the Dunn Index calculation. | [scipy.org](https://docs.scipy.org/doc/scipy/) |
| **ipywidgets** (`ipywidgets`) | Adds interactive UI controls (sliders, dropdowns, etc.) to Jupyter notebooks. We use it to create sliders for adjusting plot parameters in real time. | [ipywidgets.readthedocs.io](https://ipywidgets.readthedocs.io/en/stable/) |
| **ipympl** (`matplotlib widget` backend) | Enables interactive, live-updating Matplotlib figures inside Jupyter. Required for smooth slider updates without redrawing the entire plot. | [matplotlib.org/ipympl](https://matplotlib.org/ipympl/) |

### 3.3 Imports

All library imports for this notebook are placed in the cell below. This is a deliberate best practice: keeping all imports in one place at the top of a notebook makes it easy to see at a glance what is required, and avoids confusing errors that arise when an import cell gets skipped.

> ⚠️ **You must run the cell below before executing any other code in this notebook.** If you skip this cell, all subsequent cells will raise a `NameError`.

In [ ]:
# Listing 1.
# ─────────────────────────────────────────────────────────────────────────────
# All library imports for this notebook are placed here for clarity.
# Run this cell first before executing any code.
# ─────────────────────────────────────────────────────────────────────────────

# ── Core scientific stack ─────────────────────────────────────────────────────
import numpy as np                               # Numerical arrays and mathematical functions
import pandas as pd                              # Tabular data structures
import matplotlib.pyplot as plt                  # Plotting and visualisation
import matplotlib.patches as mpatches            # Custom legend entries in plots
import matplotlib
import seaborn as sns                            # Statistical visualisation and theming

matplotlib.rcParams['figure.max_open_warning'] = 50  # Suppress warnings when many figures are open

# ── Datasets ──────────────────────────────────────────────────────────────────
from sklearn.datasets import (
    load_breast_cancer,       # Wisconsin Breast Cancer dataset (binary classification)
    load_iris,                # Iris dataset (three-class classification)
    make_classification,      # Synthetic binary or multi-class classification data
    make_blobs,               # Synthetic isotropic Gaussian clusters
)

# ── Preprocessing and model selection ────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import train_test_split, StratifiedKFold

# ── Classifiers ───────────────────────────────────────────────────────────────
from sklearn.linear_model  import LogisticRegression
from sklearn.ensemble      import RandomForestClassifier
from sklearn.tree          import DecisionTreeClassifier
from sklearn.naive_bayes   import GaussianNB
from sklearn.neighbors     import KNeighborsClassifier
from sklearn.svm           import SVC
from sklearn.multiclass    import OneVsRestClassifier  # Extends binary classifiers to multi-class ROC

# ── Evaluation metrics ────────────────────────────────────────────────────────
from sklearn.metrics import (
    # ROC and AUC
    roc_curve, roc_auc_score, auc,
    # Precision-Recall
    precision_recall_curve, average_precision_score,
    # Threshold-based classification metrics
    confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score,
)

from sklearn.multiclass      import OneVsRestClassifier# For multi-class ROC curves

# ── Calibration ───────────────────────────────────────────────────────────────
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

# ── Clustering and unsupervised evaluation ────────────────────────────────────
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

# ── Distance utilities ────────────────────────────────────────────────────────
from scipy.spatial.distance import cdist          # Pairwise distance matrix

# ── Global random state ───────────────────────────────────────────────────────
# Fixed seed used throughout the notebook for reproducibility.
rng = np.random.default_rng(42)

print('Libraries loaded successfully.')

## 4. From Hard Predictions to Scores

🔙 [Back to Table of Contents](#table-of-contents)

Up to this point, every metric we have looked at has been based on **hard predictions**: the classifier looks at an input and commits to a single label, 0 or 1, positive or negative. That binary output is then compared against the true label and slotted into the confusion matrix. The process is straightforward, but it throws away something potentially very useful: information describing how *confident* the model was when it made that call.

Most classifiers do not actually make decisions this way internally. Under the hood, they compute a **score**, sometimes called a confidence value or a posterior probability. It represents how strongly the model believes an example belongs to the positive class. A score of 0.97 means the model is highly confident the example is positive. A score of 0.51 means it is barely nudging past the midpoint. A score of 0.12 means the model considers the example almost certainly negative. The hard label you see in the output is just the result of applying a **decision threshold** to that score. If the score is above the threshold, predict positive; if it is below, predict negative. By default that threshold is usually set at 0.5, but there is nothing magical about that value.

This matters because the right threshold is not the same in every situation. Choosing a threshold is really a question about what kinds of mistakes you can tolerate, and that depends entirely on the problem you are solving.

* In **medical screening**, missing a true case (a false negative) can be catastrophic, so you set a low threshold. You accept that some healthy individuals will be flagged unnecessarily, because the cost of missing a genuine case is far higher than the cost of an unnecessary follow-up test.

* In **spam filtering**, the calculus runs the other way. Blocking a legitimate email (a false positive) is often more disruptive than letting the occasional piece of spam through. A higher threshold protects precision at the cost of some missed spam.

* In **fraud detection**, the balance depends on transaction value, the cost of investigating a flagged case, and how much friction you are willing to add to legitimate customers' experience.

So a classifier's score-based output is not a single fixed performance point, it is a whole family of possible "operating points" depending on where you set the threshold. Evaluating a model at only one threshold, say the default 0.5, gives you an incomplete picture. The **ROC curve** is a visualisation tool that makes the full picture visible by sweeping the threshold from 0 to 1 and tracing the resulting path through a two-dimensional space defined by the true positive rate and the false positive rate. Every point on that curve corresponds to a different threshold, and the curve as a whole captures how the model behaves across *all possible operating choices at once*.

As a reminder, the true positive rate (TPR, also called recall or sensitivity) is the fraction of actual positives the model correctly identifies: 

$$\text{TPR} = \frac{TP}{TP + FN}$$

The false positive rate (FPR) is the fraction of actual negatives the model incorrectly labels as positive: 

$$\text{FPR} = \frac{FP}{FP + TN}$$

Both quantities come directly from the confusion matrix covered in Notebooks 10 and 11.

### 4.1 A Running Example: Breast Cancer Diagnosis

To make the ideas in this section easier to grasp, we'll work with a single running example. We will train a logistic regression classifier on the Wisconsin Breast Cancer dataset and use its outputs to build up the ROC curve step by step.

Remember that logistic regression is a linear classifier that models the probability that an input belongs to the positive class. Rather than drawing a hard boundary and assigning a label directly, it passes a weighted sum of the input features through a sigmoid function, which squashes the result into the range $(0, 1)$. That output is a probability, and it is exactly the kind of score we need to construct a ROC curve.

The dataset contains measurements from 569 digitised fine needle aspirate (FNA) biopsy samples. Each sample is described by 30 numerical features derived from the shape and texture of cell nuclei, and each is labelled either malignant (0) or benign (1). The task is to classify unseen samples correctly, but more than that, we want to understand *how confidently* the model makes each call, because that confidence is what the ROC curve is built from.

Once the model is trained, we do not just ask it to predict a label for each test sample. We ask it to return a **score**: the estimated probability that the sample belongs to class 1 (benign). To turn those scores into hard predictions, a **decision threshold** is applied. If the score is at or above the threshold, the sample is predicted positive (benign); if it is below, the sample is predicted negative (malignant). The default threshold of 0.5 treats the two classes symmetrically, but as we discussed above, that is rarely the right choice in practice. By trying several different thresholds, we can see directly how the trade-off between true positives and false positives shifts, and those threshold-specific operating points are exactly what the ROC curve connects into a single picture.

**Step 1. Loading the data**

We begin by loading the Wisconsin Breast Cancer dataset directly from scikit-learn. The dataset is already clean and fully numerical, so no preprocessing is needed at this stage beyond separating the features from the labels.

In [ ]:
# Listing 2.
raw_bc = load_breast_cancer()
X_bc, y_bc = raw_bc.data, raw_bc.target

print(f"Samples: {X_bc.shape[0]},  Features: {X_bc.shape[1]}")
print(f"Class counts — malignant (0): {(y_bc == 0).sum()},  benign (1): {(y_bc == 1).sum()}")

**Step 2: Split into Train and Test Sets**

Before training anything, we set aside a portion of the data as a held-out test set. The model will never see these examples during training, so the scores it produces for them give us an honest estimate of how it would behave on genuinely new data. We use `stratify=y_bc` to ensure that the class proportions in the training and test sets match those in the full dataset, which is good practice whenever the classes are not perfectly balanced.


In [ ]:
# Listing 3.
# 25% of the data is held out for evaluation.
# stratify=y_bc ensures class proportions are preserved in both splits.
X_tr, X_te, y_tr, y_te = train_test_split(
    X_bc, y_bc, test_size=0.25, random_state=0, stratify=y_bc
)

print(f"Training samples:  {X_tr.shape[0]}")
print(f"Test samples:      {X_te.shape[0]}")

**Step 3: Standardise the Features**

Logistic regression finds its decision boundary by combining feature values with learned weights. If one feature has values in the thousands and another has values between 0 and 1, the optimiser has to work much harder to find a balanced solution, and may not converge well within a fixed number of iterations. Standardising each feature to have mean 0 and standard deviation 1 puts them all on the same footing and allows the optimiser to work efficiently.

Remember, we fit the scaler on the training data only, then apply the same transformation to the test data. Fitting on the full dataset before splitting would allow information about the test set to leak into the training process, which would give an overly optimistic picture of performance. This is a subtle but important discipline to maintain.

In [ ]:
# Listing 4.
sc = StandardScaler()

# Fit the scaler on training data only, then transform both sets.
# Fitting on the full dataset before splitting would constitute data leakage.
X_tr_s = sc.fit_transform(X_tr)
X_te_s = sc.transform(X_te)

**Step 4: Train the Classifier and Extract Scores**

We now fit a logistic regression model on the standardised training data. Once trained, we do not call `predict()`, which would return hard labels. Instead we call `predict_proba()`, which returns the model's estimated probability for each class. We take column 1, the estimated probability of class 1 (benign), as our score. Every test sample now has a score between 0 and 1 representing the model's confidence that it is benign.

We then arrange those scores alongside the true labels in a DataFrame, sorted from highest score to lowest. This sorted view will be useful shortly when we step through different threshold values manually.

In [ ]:
# Listing 5.
# max_iter=2000 gives the solver enough steps to converge on this dataset
# after standardisation. The default of 100 is sometimes too few.
clf_scores = LogisticRegression(max_iter=2000, random_state=0)
clf_scores.fit(X_tr_s, y_tr)

# predict_proba returns a (n_samples, n_classes) array.
# Column 1 is P(class 1 | x), i.e. the probability of benign.
scores = clf_scores.predict_proba(X_te_s)[:, 1]

# Sort by score descending so the most confidently positive predictions
# appear at the top — useful for inspecting the threshold effect.
df_scores = pd.DataFrame({'true_label': y_te, 'score': scores})
df_scores = df_scores.sort_values('score', ascending=False).reset_index(drop=True)

**Step 5: Inspect the Score Distributions and Operating Points**

Before we draw a full ROC curve, it helps to look at two simpler pictures that build the same intuition from the ground up.

The first is a histogram of the scores, split by true class. If the model has learned something useful, the scores for benign samples should pile up near 1 and the scores for malignant samples should pile up near 0. The more cleanly the two distributions separate, the better the classifier, and the more room we have to choose a threshold that works well for our application.

The second plot shows what happens when we pick three specific threshold values and plot the resulting (FPR, TPR) pair for each one as a point in ROC space. Each point represents one operating choice. A low threshold (0.3) calls many samples positive, pushing TPR up but also pulling FPR up. A high threshold (0.7) is more cautious, keeping FPR low but missing more true positives. The diagonal dashed line is the line of no discrimination: a model that assigns scores at random would produce operating points scattered along it, because its TPR and FPR would be equal at every threshold.

What we are about to plot are just three isolated points. The ROC curve, which we construct in the next section, connects every such point across every possible threshold into a single continuous picture.

In [ ]:
# Listing 6.
# Create the figure...
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

# Left: score distributions per class.
# A well-separated pair of histograms indicates the model assigns
# consistently higher scores to true positives than to true negatives.
ax = axes[0]
ax.hist(df_scores[df_scores['true_label'] == 1]['score'],
        bins=30, color='steelblue', alpha=0.6, edgecolor='white', lw=0.4,
        label='Benign (1)')
ax.hist(df_scores[df_scores['true_label'] == 0]['score'],
        bins=30, color='tomato', alpha=0.6, edgecolor='white', lw=0.4,
        label='Malignant (0)')
ax.set_xlabel('Predicted probability of Benign')
ax.set_ylabel('Count')
ax.set_title('Score distributions per class\n'
             'Good model: classes separate well in score space')
ax.legend(fontsize=9)

# Right: three specific threshold choices plotted as points in ROC space.
# This previews the ROC curve before we draw the full version.
ax = axes[1]

THRESHOLDS_DEMO = [0.3, 0.5, 0.7]
COLOURS_THRESH  = ['steelblue', 'black', 'tomato']

for t, col in zip(THRESHOLDS_DEMO, COLOURS_THRESH):
    y_pred_t = (scores >= t).astype(int)
    tpr_t = recall_score(y_te, y_pred_t)
    # FPR = 1 - specificity = 1 - recall computed on the negative class
    fpr_t = 1 - recall_score(y_te, y_pred_t, pos_label=0)
    ax.scatter(fpr_t, tpr_t, s=120, color=col, edgecolors='k', lw=0.8,
               zorder=5, label=f'threshold={t}  TPR={tpr_t:.2f}  FPR={fpr_t:.2f}')

# The diagonal represents a random classifier with no discriminative power.
ax.plot([0, 1], [0, 1], 'k--', lw=1.2, alpha=0.4, label='Random classifier')
ax.set_xlabel('FPR (False Positive Rate)')
ax.set_ylabel('TPR (True Positive Rate / Recall)')
ax.set_title('Three operating points in ROC space\n'
             'Lower threshold → up and right (more TP, more FP)')
ax.legend(fontsize=8)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

### 4.2 Building the ROC Curve

The ROC curve is built by asking a simple question at every possible decision threshold: if I draw the line here, what fraction of true positives do I catch, and what fraction of true negatives do I accidentally flag? Scikit-learn's `roc_curve()` function answers that question for every unique score value in one call, returning the full set of (FPR, TPR) pairs that trace the curve from bottom-left to top-right.

The two functions you need are both in `sklearn.metrics`:

```python
from sklearn.metrics import roc_curve, roc_auc_score
```

`roc_curve(y_true, y_score)` takes the true binary labels and the predicted scores (probabilities or any other continuous confidence value) and returns three parallel arrays: the false positive rates, the true positive rates, and the threshold value that produced each pair. You can apply this to the logistic regression classifier we trained above, or to any other scikit-learn classifier that exposes `predict_proba()` or `decision_function()`.

`roc_auc_score(y_true, y_score)` takes the same inputs and returns the area under the ROC curve as a single number. This is a useful summary of overall classifier quality that does not require you to commit to a threshold.

Reminder: the two metrics that define the axes of the ROC curve are:

$$\text{TPR} = \frac{TP}{TP + FN} \qquad \text{FPR} = \frac{FP}{FP + TN}$$

TPR (true positive rate, also called recall or sensitivity) measures how many of the actual positives the model catches. FPR (false positive rate, equal to $1 - \text{specificity}$) measures how many of the actual negatives are incorrectly called positive. A perfect classifier sits at the top-left corner of the plot, where TPR is 1 and FPR is 0: it catches every positive while raising no false alarms. A random classifier produces operating points scattered along the diagonal, where TPR and FPR are equal at every threshold because the scores carry no information about the true class.

The code below computes the ROC curve for the logistic regression classifier we trained in Section 4.1 and plots the result.

In [ ]:
# Listing 7.
# roc_curve() sweeps the decision threshold across every unique score value
# in descending order, computing the (FPR, TPR) pair at each step.
# It returns three parallel arrays:
#   fpr_sk    : false positive rate at each threshold
#   tpr_sk    : true positive rate at each threshold
#   thresh_sk : the threshold value that produced each (FPR, TPR) pair
fpr_sk, tpr_sk, thresh_sk = roc_curve(y_te, scores)

# roc_auc_score() computes the area under the ROC curve directly from the
# scores, without needing to choose a threshold. A value of 1.0 means
# perfect separation; 0.5 means the model is no better than random.
auc_sk = roc_auc_score(y_te, scores)

fig, ax = plt.subplots(figsize=(7, 6))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

# The ROC curve itself: each point is one (FPR, TPR) pair corresponding
# to a specific threshold. Moving left along the curve means raising the
# threshold (more cautious); moving right means lowering it (more permissive).
ax.plot(fpr_sk, tpr_sk, color='steelblue', lw=2.5,
        label=f'Logistic Regression (AUC = {auc_sk:.3f})')

# The diagonal represents a classifier that assigns scores at random.
# Any useful model should produce a curve that bows above this line.
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random classifier (AUC = 0.50)')

# Shading the area under the curve gives a visual impression of AUC magnitude.
ax.fill_between(fpr_sk, tpr_sk, alpha=0.10, color='steelblue',
                label='Area under curve (AUC)')

# The top-left corner (FPR=0, TPR=1) is where a perfect classifier sits:
# it catches every positive while producing zero false alarms.
ax.scatter([0], [1], s=200, color='gold', marker='*', edgecolors='k',
           lw=1.0, zorder=6, label='Perfect classifier')

ax.set_xlabel('FPR  (False Positive Rate)  =  1 − Specificity')
ax.set_ylabel('TPR  (True Positive Rate)  =  Recall')
ax.set_title('Figure 1: ROC curve\nEach point corresponds to one decision threshold')
ax.legend(fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

---

### 4.3 Generating ROC Curves for Other Classifiers

The code in Section 4.2 is not specific to logistic regression. The same `roc_curve()` function works with any scikit-learn classifier, because it only requires a set of true labels and a corresponding set of scores. The classifier itself does not matter, only the scores it produces.

The pattern is always the same: train the model on the standardised training data, extract scores for the positive class from the test data, then pass those scores to `roc_curve()`. The only thing that varies between classifiers is how you obtain the scores.

Most classifiers expose `predict_proba()`, which returns an array of shape `(n_samples, n_classes)`. Column 1 gives the estimated probability of the positive class, and that is the score you pass in:

```python
scores = clf.predict_proba(X_te_s)[:, 1]
fpr, tpr, thresholds = roc_curve(y_te, scores)
```

Some classifiers, most notably support vector machines, do not produce calibrated probabilities by default. Instead they expose `decision_function()`, which returns a raw confidence value for each sample. This is still a perfectly valid input to `roc_curve()` because the ROC curve depends only on the *ranking* of scores, not their absolute values or whether they lie between 0 and 1:

```python
scores = clf.decision_function(X_te_s)
fpr, tpr, thresholds = roc_curve(y_te, scores)
```

Once you have the arrays returned by `roc_curve()`, plotting the curve for a different classifier is identical to what we did in Section 4.2. 

### 4.4 The Area Under the Curve (AUC)

The ROC curve gives a complete picture of how a classifier behaves across all possible thresholds, but comparing two curves visually can be difficult, especially when they cross. It would be useful to have a single number that summarises the overall quality of the curve without requiring you to commit to a threshold. That number is the **AUC**, the area under the ROC curve.

Because the ROC curve lives inside a unit square (both axes run from 0 to 1), the AUC is always a value between 0 and 1. A curve that hugs the top-left corner encloses a large area; a curve that follows the diagonal encloses exactly half the square.

The AUC has a clean probabilistic interpretation that makes it easy to reason about:

$$\text{AUC} = P(\text{score of a random positive} > \text{score of a random negative})$$

In plain terms, if you picked one true positive and one true negative at random from your test set, the AUC is the probability that the model assigned the higher score to the positive. A perfect classifier always ranks positives above negatives, so its AUC is 1.0. A classifier that assigns scores at random has no systematic tendency to rank positives higher, so its AUC converges to 0.5.

| AUC | Interpretation |
|-----|----------------|
| 1.0 | Perfect classifier: every positive outranks every negative |
| 0.5 | Random guessing: scores carry no information about the true class |
| < 0.5 | Worse than random: scores are inverted (reversing the decision rule would improve things) |

This interpretation is entirely threshold-free, which makes AUC a fair basis for comparing classifiers that may have been trained with different default thresholds or evaluated on datasets with different class proportions.

Scikit-learn computes the AUC directly from the scores with `roc_auc_score()`:

```python
from sklearn.metrics import roc_auc_score

# roc_auc_score() takes the true binary labels and the predicted scores.
# It does not require a threshold: it works from the full ranking of scores.
auc = roc_auc_score(y_te, scores)

print(f'AUC: {auc:.4f}')
```

In practice you will usually compute the ROC curve and the AUC together and include the AUC value in the legend of the plot, as we did in Section 4.2, so that the curve and its summary are always seen side by side.

> ⚠️ **Warning:** AUC summarises performance equally across all thresholds, including many that would never be used in practice. A higher AUC does not guarantee better performance at the specific operating point your application requires. Always inspect the curve itself in the region of FPR values that are acceptable for your use case, and treat AUC as a useful first filter rather than the final word on which model to prefer.

---

Figure 2 lets you interact with a real ROC curve. Rather than working with abstract fractions, it uses a fixed dataset of 10,000 examples so you can see the actual counts of true positives, false positives, true negatives, and false negatives that each threshold choice produces. Drag the slider and watch the red dot move along the curve while the confusion matrix updates in the panel alongside it.

A few positions are worth exploring deliberately. At a low threshold, TPR climbs close to 1 but FPR rises with it: the classifier catches almost everything but flags a large number of negatives along the way. At a high threshold, FPR falls close to zero but TPR drops too: very few false alarms, but many genuine positives are missed. Somewhere in between lies whatever balance your application requires.

The blue curve and the AUC value never change as you move the slider. Both are properties of the classifier itself, not of any particular threshold. The slider only controls where on that curve you choose to operate.

In [ ]:
# Listing 8.
%matplotlib widget
from visualisations.Figure_2 import show
show()

---

## 5. Reading the ROC Curve

The AUC gives you a single number, but the shape of the ROC curve tells a richer story. Two classifiers can have similar AUC values yet behave very differently across the threshold range, and some curve shapes carry specific warnings that the number alone would hide. Figure 3 shows five patterns that appear frequently in practice.

* **Perfect (AUC = 1.0).** The curve goes straight up the left edge and across the top. Every positive example scores higher than every negative, so there exists a threshold that catches all positives while producing zero false alarms. In practice this almost never happens on real data and is usually a sign of data leakage if you see it.

* **Good (AUC > 0.5).** The curve bows toward the top-left corner. The model assigns higher scores to most positives than to most negatives, and the further the bow, the better the separation. This is what a well-trained classifier on realistic data looks like.

* **Random (AUC = 0.5).** The curve follows the diagonal. At every threshold, the fraction of true positives caught equals the fraction of false positives introduced, which means the scores carry no information about the true class. The model is doing no better than guessing.

* **Poor / Inverted (AUC < 0.5).** The curve bends below the diagonal. This looks bad but is not necessarily a useless model: it means the scores are inverted, positives are receiving low scores and negatives are receiving high ones. Flipping the decision rule (predict positive when the score is below the threshold rather than above it) recovers an effective AUC of $1 - \text{original AUC}$, which may be quite good.

* **Misleading (high AUC, tiny minority class).** The curve bows upward and the AUC looks healthy, but the positive class contains only a handful of examples. A small number of minority-class points can pull the curve toward the top-left even when the model would perform poorly in deployment, where the imbalance matters. A high AUC on a severely imbalanced dataset should always prompt you to look at the precision-recall curve as well, which we cover in Section 6.

> ⚠️ **Warning:** A perfect or near-perfect AUC on your training data almost always indicates data leakage: information from the test set has found its way into the training process, either through the features themselves or through preprocessing steps applied before the train-test split. Always check your pipeline if you see AUC values above roughly 0.99 on real-world data.

In [ ]:
# Listing 9.
%matplotlib widget
from visualisations.Figure_3 import show
show()

### 5.1 Comparing Multiple Classifiers

One of the most practical uses of the ROC curve is placing several classifiers on the same axes and comparing them directly. Because the curve is threshold-free, you are comparing the models themselves rather than any particular operating choice, and the visual immediately tells you more than a table of numbers would.

The mechanics are straightforward. For each classifier, fit the model, extract scores with `predict_proba()`, and call `roc_curve()` and `roc_auc_score()` exactly as we did in Section 4.2. Then plot each curve on the same axes:

```python
for name, clf in classifiers:
    clf.fit(X_tr_s, y_tr)
    scores = clf.predict_proba(X_te_s)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, scores)
    auc = roc_auc_score(y_te, scores)
    ax.plot(fpr, tpr, label=f'{name}  (AUC = {auc:.3f})')
```

The resulting plot supports two kinds of comparison.

**When one curve lies entirely above another**, the upper classifier dominates: it achieves a higher TPR than the lower one at every possible FPR value, meaning it is strictly better regardless of where you set the threshold. In this case the AUC ranking is a reliable summary of the difference.

**When curves cross**, neither classifier dominates. One may perform better in the low-FPR region (where you need high specificity, as in medical screening) while the other performs better at higher FPR values (where catching more positives matters more, even at the cost of more false alarms). The global AUC averages over the entire curve and cannot tell you which model is better for your specific operating region. This is why it is worth zooming into the part of FPR space your application actually uses.

The code below fits five classifiers on the breast cancer dataset and produces two panels: the full ROC plot on the left, and a zoomed view of the low-FPR region (0 to 0.15) on the right. The zoom panel often reveals a different ranking from the one suggested by the global AUC scores printed below the plot.

In [ ]:
# Listing 10.
# Five classifiers fitted on the same breast cancer train/test split.
# Using predict_proba() on each so we can pass continuous scores to roc_curve().
# max_depth=4 on the decision tree prevents it overfitting to noise, which
# would make its ROC curve artificially jagged.
CLASSIFIERS_ROC = [
    ('Logistic Reg.',  LogisticRegression(max_iter=2000, random_state=0)),
    ('Random Forest',  RandomForestClassifier(n_estimators=100, random_state=0)),
    ('Decision Tree',  DecisionTreeClassifier(max_depth=4, random_state=0)),
    ('Naive Bayes',    GaussianNB()),
    ('k-NN (k=5)',     KNeighborsClassifier(n_neighbors=5)),
]
COLOURS_ROC = ['steelblue', 'seagreen', 'tomato', 'goldenrod', 'mediumpurple']

fig, axes = plt.subplots(1, 2, figsize=(10, 6))
fig.canvas.header_visible  = False
fig.canvas.toolbar_visible = False

for ax in axes:
    ax.plot([0, 1], [0, 1], 'k--', lw=1.2, alpha=0.4, label='Random (AUC = 0.50)')
    ax.set_ylabel('TPR')
    ax.grid(True, alpha=0.25)

auc_scores = {}

for (name, clf), col in zip(CLASSIFIERS_ROC, COLOURS_ROC):
    clf.fit(X_tr_s, y_tr)
    sc_clf = clf.predict_proba(X_te_s)[:, 1]
    fpr_c, tpr_c, _ = roc_curve(y_te, sc_clf)
    auc_c = roc_auc_score(y_te, sc_clf)
    auc_scores[name] = auc_c

    # Plot on both panels with the same colour and label; the zoom panel
    # will clip the x-axis to reveal how relative rankings shift at low FPR.
    axes[0].plot(fpr_c, tpr_c, color=col, lw=2.2, label=f'{name} (AUC = {auc_c:.3f})')
    axes[1].plot(fpr_c, tpr_c, color=col, lw=2.2, label=name)

axes[0].set_xlabel('FPR')
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1.02)
axes[0].set_title('Figure 4: ROC curves — multiple classifiers\n'
                  'A curve above another dominates at all thresholds')
axes[0].legend(fontsize=8)

# Zooming to FPR 0–0.15 isolates the high-specificity operating region,
# where applications such as medical screening typically need to operate.
# Classifier rankings often change here relative to the global AUC ranking.
axes[1].set_xlabel('FPR (zoomed: 0 to 0.15)')
axes[1].set_xlim(0, 0.15)
axes[1].set_ylim(0, 1.02)
axes[1].set_title('Figure 5: Zoomed — low FPR region\n'
                  'Rankings can change depending on the operating region')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print('AUC scores (global ranking):')
for name, auc_c in sorted(auc_scores.items(), key=lambda x: x[1], reverse=True):
    print(f'  {name:<22}  AUC = {auc_c:.4f}')

---

All five classifiers perform well on this dataset, with AUC values above 0.92. Logistic regression leads by a clear margin, followed closely by random forest and naive Bayes. The decision tree sits noticeably lower, which is expected: even with `max_depth=4` limiting its complexity, a single tree produces a coarser decision boundary than the ensemble and probabilistic methods above it.

These global rankings may not hold in the low-FPR region, however. Check the zoomed panel on the right of Figure 4: if any curves cross below FPR = 0.15, the ranking there is what matters for high-specificity applications, not the numbers printed above.

---

## 6. Multi-Class ROC Curves

🔙 [Back to Table of Contents](#table-of-contents)

The ROC curve is defined for binary classification: one positive class, one negative class. When your problem has more than two classes, you need a strategy for decomposing the multi-class problem into a set of binary ones. Two approaches are standard.

### 6.1 One-vs-Rest (OvR)

In the One-vs-Rest strategy, you iterate over each class $c$ in turn, treating it as the positive class and grouping all remaining classes together as the negative class. This produces $n$ separate binary problems and $n$ separate ROC curves, one per class.

Scikit-learn handles this with `label_binarize()`, which converts a multi-class label array into a binary indicator matrix, and `OneVsRestClassifier`, which wraps any classifier to apply the OvR decomposition automatically:

```python
from sklearn.preprocessing import label_binarize
from sklearn.multiclass    import OneVsRestClassifier

# Convert integer class labels to a binary indicator matrix.
# classes= must list every class present in the dataset.
# For a three-class problem this produces an (n_samples, 3) array
# where each column is 1 if that sample belongs to that class, 0 otherwise.
y_bin = label_binarize(y, classes=[0, 1, 2])

# Wrap the classifier so it fits one binary model per class automatically.
clf_ovr = OneVsRestClassifier(LogisticRegression(max_iter=1000))
clf_ovr.fit(X_tr_s, y_tr)

# predict_proba returns an (n_samples, n_classes) array.
# Each column is the score for one class against all others.
y_score = clf_ovr.predict_proba(X_te_s)
```

Once you have the per-class scores, computing ROC curves and AUC values follows exactly the same pattern as in the binary case, applied once per class:

```python
from sklearn.metrics import roc_curve, roc_auc_score

for i, class_name in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_bin_te[:, i], y_score[:, i])
    auc = roc_auc_score(y_bin_te[:, i], y_score[:, i])
```

The per-class AUC values can then be aggregated into a single summary number in three ways, each making a different assumption about how much the classes matter relative to one another:

| Aggregation | How it is computed | When to use it |
|---|---|---|
| **Macro AUC** | Simple arithmetic mean of the $n$ per-class AUC values | When all classes are equally important regardless of how many examples each has |
| **Weighted AUC** | Mean of per-class AUC values weighted by the number of true instances in each class | When class sizes differ and larger classes should carry more influence |
| **Micro AUC** | Pools all TP, FP, and FN counts across classes before computing a single ROC curve | Dominated by the largest class; useful when overall instance-level accuracy matters |

In scikit-learn, `roc_auc_score()` computes all three via its `average` parameter:

```python
# multi_class='ovr' tells roc_auc_score to use the One-vs-Rest strategy.
# average controls how per-class AUC values are combined.
macro_auc    = roc_auc_score(y_bin_te, y_score, multi_class='ovr', average='macro')
weighted_auc = roc_auc_score(y_bin_te, y_score, multi_class='ovr', average='weighted')
```

Note that micro AUC is not directly supported via `roc_auc_score()` in the multi-class setting. To compute it you need to construct the micro-averaged ROC curve manually by concatenating the per-class binary arrays and calling `roc_curve()` on the combined result, which the code below demonstrates.

### 6.2 One-vs-One (OvO)

The One-vs-One strategy trains a separate binary classifier for every pair of classes $(c_i, c_j)$, ignoring all other classes during each pairwise comparison. For $n$ classes this produces $\binom{n}{2}$ classifiers and curves, which grows quickly: three classes give three curves, ten classes give 45. OvO is supported in `roc_auc_score()` via `multi_class='ovo'` and is sometimes preferred when class confusion between specific pairs is the primary concern, but its computational cost makes it impractical for problems with many classes.

```python
# OvO AUC requires the raw predicted probabilities, not binarised labels.
ovo_auc = roc_auc_score(y_te, y_score, multi_class='ovo', average='macro')
```

### 6.3 A Key Limitation of OvR

Each per-class OvR curve asks whether class $c$ is separable from all other classes combined. A class can achieve a high per-class AUC while still being frequently confused with one specific neighbouring class, because the errors it makes against that neighbour are diluted by the many correct rejections of all other classes. High per-class AUC values in an OvR decomposition do not guarantee that the classifier can reliably distinguish between every pair of classes. If pairwise confusion is a concern in your problem, inspect the confusion matrix alongside the ROC curves rather than relying on the aggregated AUC alone.

---

The code below applies the OvR strategy to a synthetic three-class dataset that has been deliberately constructed to be difficult. Eight features are generated, of which only four carry genuine class signal and two are redundant linear combinations of those four. The remaining two are pure noise. Each class forms two separate clusters in feature space rather than one, creating non-convex, overlapping regions that a linear classifier cannot cleanly separate. On top of that, 5% of the labels are randomly flipped to simulate the kind of annotation noise common in real datasets. The result is a problem where meaningful but imperfect separation is the best any linear model can achieve, which is exactly the condition under which the ROC curve is most informative.

The OvR decomposition produces three binary problems: Class 0 against everything else, Class 1 against everything else, and Class 2 against everything else. Each problem gets its own ROC curve and its own AUC value. The macro AUC printed in the title averages these three values equally, giving an overall picture of how well logistic regression separates the three classes in aggregate.

A few things are worth looking for in the plot. If one class has a noticeably higher AUC than the others, it means that class is more linearly separable from the rest of the dataset than its neighbours are, perhaps because its feature distribution overlaps less with the combined negative pool. If all three curves cluster together at moderate AUC values, the classes are genuinely hard to separate from one another and no single class is easier to identify than the rest. If any curve dips close to the diagonal, that class's scores carry almost no signal when it is treated as the positive class under OvR, which could indicate heavy overlap with one or more of the other classes or an insufficiently expressive model.

> ⚠️ **Warning:** A moderate macro AUC does not mean every pair of classes is equally confusable. OvR compares each class against all others combined, which can mask pairwise confusion. If Class 1 and Class 2 are frequently confused with each other but both separate cleanly from Class 0, all three per-class AUC values may still look reasonable. Always examine the confusion matrix alongside the ROC curves to check whether errors are concentrated between specific class pairs.

In [ ]:
# Listing 11.
# ── Generate a deliberately difficult three-class dataset ─────────────────────
# n_informative=4 means only 4 of the 8 features carry class signal;
# the rest are noise. n_clusters_per_class=2 creates non-convex, overlapping
# class regions that are hard for a linear classifier to separate cleanly.
# flip_y=0.05 randomly mislabels 5% of examples, simulating real-world noise.
X, y = make_classification(
    n_samples=2000,
    n_features=8,
    n_informative=4,
    n_redundant=2,
    n_classes=3,
    n_clusters_per_class=2,
    flip_y=0.05,
    random_state=42,
)

CLASSES = ['Class 0', 'Class 1', 'Class 2']

# Binarise labels into a (n_samples, 3) indicator matrix, one column per class.
y_bin = label_binarize(y, classes=[0, 1, 2])

X_tr, X_te, y_tr, y_te_bin = train_test_split(
    X, y_bin, test_size=0.25, random_state=0, stratify=y
)

sc     = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s = sc.transform(X_te)

# ── Fit logistic regression under the OvR strategy ───────────────────────────
clf = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=0))
clf.fit(X_tr_s, y_tr)

# predict_proba returns (n_samples, n_classes); column i = score for class i.
y_score = clf.predict_proba(X_te_s)

# ── Plot one ROC curve per class ──────────────────────────────────────────────
COLOURS = ['steelblue', 'tomato', 'seagreen']

fig, ax = plt.subplots(figsize=(7, 6))
fig.canvas.header_visible  = False
fig.canvas.toolbar_visible = False

ax.plot([0, 1], [0, 1], 'k--', lw=1.2, alpha=0.4, label='Random (AUC = 0.50)')

for i, (class_name, col) in enumerate(zip(CLASSES, COLOURS)):
    fpr, tpr, _ = roc_curve(y_te_bin[:, i], y_score[:, i])
    auc = roc_auc_score(y_te_bin[:, i], y_score[:, i])
    ax.plot(fpr, tpr, color=col, lw=2.2, label=f'{class_name}  (AUC = {auc:.3f})')

macro_auc = roc_auc_score(y_te_bin, y_score, multi_class='ovr', average='macro')

ax.set_xlabel('FPR  (False Positive Rate)')
ax.set_ylabel('TPR  (True Positive Rate)')
ax.set_title(f'Figure 6: OvR ROC curves — three-class synthetic dataset\n'
             f'Macro AUC = {macro_auc:.3f}')
ax.legend(fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()



---

## 7. Weaknesses of ROC Curves

🔙 [Back to Table of Contents](#table-of-contents)

ROC curves and AUC are powerful and widely used, but every practitioner needs to know where they can mislead. There are three failure modes worth understanding before relying on them in a real application.

**Weakness 1: insensitivity to class imbalance.** FPR is defined as $\text{FPR} = FP / (FP + TN)$. When the negative class is very large, the denominator is dominated by a huge number of true negatives, and even a large absolute count of false positives barely moves the needle. A model that fails to catch most of the rare positive cases can still trace a curve that looks healthy on the ROC plot, simply because its false positive rate stays low by arithmetic rather than by skill.

**Weakness 2: AUC averages over thresholds you will never use.** AUC summarises the entire curve equally from FPR = 0 to FPR = 1. In deployment you commit to a single threshold, and the part of the curve that matters is a small region around that operating point. A model with a higher global AUC can be outperformed by a competitor at the specific threshold your application requires.

**Weakness 3: crossing curves hide operating-point differences.** When two ROC curves cross, neither model dominates the other. The model with the higher global AUC may perform worse in exactly the region of FPR space your application operates in. The summary number obscures this entirely.

Figures 7 and 8 demonstrate all three weaknesses on a synthetic dataset with a 95/5 class split. Two models, A and B, are constructed with deliberately similar global AUC values. The printed table shows what is actually happening at the default threshold of 0.5:

```
  Model          AUC    TP    FP    FN    TN    Recall
  -------------------------------------------------------
  Model A      0.966    46   105     4   845      0.92
  Model B      0.976    24     2    26   948      0.48
```

Model B has the higher AUC, yet at threshold 0.5 it catches only 24 of the 50 true positives, a recall of 0.48. Model A catches 46 of them (recall 0.92), at the cost of more false positives. On a balanced dataset those false positives would push up FPR and the ROC curve would reflect the trade-off clearly. On this imbalanced dataset, 105 false positives out of 950 negatives is only an FPR of 0.11, which looks perfectly reasonable on the global plot.

Figure 7 shows the full ROC curves for both models. They are close enough that the global AUC difference of 0.01 is not obviously meaningful. Figure 8 zooms into the low-FPR region (0 to 0.10) and adds a reference line at FPR = 0.05. Here the curves cross: Model A, with the lower global AUC, outperforms Model B in the high-specificity region to the left of the crossing point. A practitioner choosing between these two models on the basis of AUC alone would pick the worse one for any application that requires operating below FPR = 0.05.

> ⚠️ **Warning:** On imbalanced datasets, always examine recall and precision at your intended operating threshold alongside the ROC curve. The Precision-Recall curve, introduced in Section 8, is specifically designed for situations where the positive class is rare and recall is what matters most.

In [ ]:
# Listing 12.
%matplotlib widget
from visualisations.Figure_7 import show
show()


---

## 8. Precision-Recall Curves

🔙 [Back to Table of Contents](#table-of-contents)

The fundamental weakness of the ROC curve on imbalanced data comes down to FPR. Because FPR divides by the total number of negatives, a large negative class absorbs false positives and keeps FPR artificially low even when the model is producing a damaging number of them in absolute terms. The Precision-Recall curve fixes this by removing the negative class from both axes entirely.

The x-axis is Recall (how many of the true positives the model catches) and the y-axis is Precision (of everything the model calls positive, how many actually are):

$$\text{Recall} = \frac{TP}{TP + FN} \qquad \text{Precision} = \frac{TP}{TP + FP}$$

Precision has $TP + FP$ in the denominator, not $TN$. This means every false positive directly and immediately reduces precision, regardless of how many true negatives exist. There is nowhere for false positives to hide. On an imbalanced dataset this is exactly the property you need: a model that achieves high recall by flagging everything positive will see its precision collapse toward the class prevalence rate, exposing the strategy rather than rewarding it.

**Average Precision (AP)** summarises the PR curve as a single number, computed as the weighted mean of precisions across all thresholds:

$$AP = \sum_n (R_n - R_{n-1}) \, P_n$$

where $P_n$ and $R_n$ are the precision and recall at the $n$-th threshold. The critical difference from AUC is the random baseline. For the ROC curve, a random classifier gives AUC = 0.5 regardless of class balance. For the PR curve, the random baseline equals the proportion of positives in the dataset. On a 95/5 split, a random classifier achieves AP ≈ 0.05, which means an AP of 0.70 is genuinely impressive rather than just comfortably above 0.5.

---

Figures 9-11 below place the ROC and PR curves for the same two models from Section 7 side by side, so the contrast is direct. Panel 1 shows the ROC curves: both models look reasonable and the curves are close together, with AUC values of 0.966 and 0.976. Panel 2 shows the PR curves for the same models on the same 95/5 dataset: the gap between them is now clearly visible against a random baseline sitting at precision = 0.05. This is the PR curve doing its job, surfacing a real difference that the ROC curve obscured.

The AP values also tell a more nuanced story than the recall figures from Section 7 might suggest. Model B, despite catching far fewer positives at the default threshold, has a higher AP (0.829 vs 0.703). This is because its negatives score very low, so when it does call something positive it tends to be right: high precision at the cost of low recall. Model A casts a wider net, achieving high recall but at the cost of many false positives that drag precision down. Neither model is universally better; which one to prefer depends on whether your application penalises missed positives or false alarms more heavily.

Panel 3 shows PR curves for three classifiers on a balanced synthetic dataset, for contrast. When the classes are roughly equal in size, a good classifier keeps precision high across the full recall range and the curve stays well above the random baseline at 0.5. This is what a healthy PR curve looks like under normal class proportions.

> 💡 **Intuition:** Use ROC curves when classes are roughly balanced and you want a threshold-free summary of overall separability. Switch to PR curves when one class is rare and performance on that minority class is what matters. The PR curve cannot be flattered by a large negative class, which makes it the more honest tool in imbalanced settings.

In [ ]:
# Listing 13.
%matplotlib widget
from visualisations.Figure_9 import show
show()

### 8.1 Computing PR Curves in Scikit-Learn

The two functions you need are both in `sklearn.metrics`:

```python
from sklearn.metrics import precision_recall_curve, average_precision_score
```

`precision_recall_curve(y_true, probas_pred)` takes the true binary labels and the predicted scores and returns three parallel arrays: precision values, recall values, and the threshold that produced each pair. The curve is swept from high threshold to low, so it runs from high precision and low recall at the left, to low precision and high recall at the right.

```python
prec, rec, thresholds = precision_recall_curve(y_te, scores)
```

Note that `precision_recall_curve()` returns one more precision and recall value than it does thresholds. The final point is always precision = 1.0, recall = 0.0, which corresponds to a threshold so high that nothing is labelled positive. This is appended automatically and has no matching threshold entry, so `len(thresholds) == len(prec) - 1`.

`average_precision_score(y_true, y_score)` computes the area under the PR curve directly from the scores without requiring you to call `precision_recall_curve()` first:

```python
ap = average_precision_score(y_te, scores)
print(f'Average Precision: {ap:.4f}')
```

Plotting the curve follows the same pattern as the ROC curve. The key difference is axis orientation: recall goes on the x-axis and precision on the y-axis, and the random baseline is a horizontal line at the positive class prevalence rather than a diagonal:

```python
import matplotlib.pyplot as plt

# The random baseline sits at the positive class prevalence.
# A useful model must lift substantially above this line.
baseline = y_te.mean()

prec, rec, _ = precision_recall_curve(y_te, scores)
ap = average_precision_score(y_te, scores)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(rec, prec, color='steelblue', lw=2.5, label=f'Classifier (AP = {ap:.3f})')
ax.axhline(baseline, color='black', lw=1.5, linestyle='--',
           label=f'Random baseline (AP = {baseline:.2f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()
```

The same pattern works for any scikit-learn classifier that exposes `predict_proba()`. For classifiers that expose `decision_function()` instead, such as SVMs, pass the raw decision scores directly: `precision_recall_curve()` depends only on the ranking of scores, not their absolute values, in exactly the same way as `roc_curve()`.

> ⚠️ **Warning:** Unlike the ROC curve, the PR curve is sensitive to class imbalance in its baseline. An AP of 0.80 means something very different on a balanced dataset (random baseline = 0.50) than on a 95/5 dataset (random baseline = 0.05). Always report the class prevalence alongside the AP so the number can be interpreted correctly.

### 8.2 Multi-Class PR Curves

PR curves extend to multi-class problems using exactly the same One-vs-Rest strategy introduced in Section 6. Each class is treated as the positive class in turn, all other classes are grouped as negative, and a separate PR curve and AP value are computed for each. The only addition is that each panel needs its own random baseline, because the baseline is the prevalence of that specific class in the test set, and that prevalence will differ between classes whenever the dataset is not perfectly balanced.

```python
from sklearn.metrics import precision_recall_curve, average_precision_score

for i, class_name in enumerate(class_names):
    # y_bin_te[:, i] is the binary indicator for class i (1 if positive, 0 otherwise).
    # y_score[:, i] is the predicted probability of class i from the OvR classifier.
    prec, rec, _ = precision_recall_curve(y_bin_te[:, i], y_score[:, i])
    ap = average_precision_score(y_bin_te[:, i], y_score[:, i])

    # The random baseline for this class equals its prevalence in the test set.
    baseline = y_bin_te[:, i].mean()
```

The dotted horizontal lines in the plot below mark that per-class random baseline. A curve that barely lifts above its baseline is a sign that the OvR decomposition is struggling to separate that class from the combined pool of all others, even though the overall AP value may look moderate. Comparing how far each class's curve sits above its own baseline is a more honest measure of per-class performance than comparing AP values across classes directly, because those AP values are anchored to different baselines.

In [ ]:
# Listing 14.
# The dataset, binarised labels, train/test split, scaler, and fitted OvR
# classifier are reused from the ROC example above. Only the metric
# functions and plot change.
from sklearn.metrics import precision_recall_curve, average_precision_score

COLOURS = ['steelblue', 'tomato', 'seagreen']

fig, ax = plt.subplots(figsize=(7, 6))
fig.canvas.header_visible  = False
fig.canvas.toolbar_visible = False

for i, (class_name, col) in enumerate(zip(CLASSES, COLOURS)):
    prec, rec, _ = precision_recall_curve(y_te_bin[:, i], y_score[:, i])
    ap = average_precision_score(y_te_bin[:, i], y_score[:, i])

    # Random baseline for this class equals its prevalence in the test set.
    # Each class has its own baseline because class sizes may differ.
    baseline = y_te_bin[:, i].mean()

    ax.plot(rec, prec, color=col, lw=2.2,
            label=f'{class_name}  (AP = {ap:.3f},  baseline = {baseline:.2f})')
    ax.axhline(baseline, color=col, lw=1.0, linestyle=':', alpha=0.7)

# Macro AP: simple average of the three per-class AP values.
# average_precision_score accepts the full binarised label matrix and score
# matrix directly; average='macro' weights all classes equally.
macro_ap = average_precision_score(y_te_bin, y_score, average='macro')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title(f'Figure 12: OvR PR curves — three-class synthetic dataset\n'
             f'Macro AP = {macro_ap:.3f}')
ax.legend(fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

The code above reuses everything already computed in the ROC example: the same dataset, the same train/test split, the same fitted OvR logistic regression model, and the same `y_score` probability matrix. The only thing that changes is which metric functions are called. This is deliberate: placing the ROC and PR curves side by side on the same underlying model makes the comparison meaningful. Any difference you see between the two plots is a property of the evaluation tool, not of the classifier.

The loop follows an identical structure to the ROC version. For each class `i`, `precision_recall_curve()` receives the binary indicator column `y_te_bin[:, i]` as the true labels and the corresponding column of `y_score` as the predicted scores. It returns precision and recall arrays that trace the curve from high threshold (high precision, low recall) to low threshold (low recall, high precision). The `average_precision_score()` call on the same inputs returns the area under that curve as a single number.

The one addition that has no equivalent in the ROC code is the per-class baseline. Each dotted horizontal line sits at `y_te_bin[:, i].mean()`, which is the proportion of test examples that belong to class `i`. This is the AP a random classifier would achieve for that class, and it varies between classes whenever the dataset is not perfectly balanced. Reading the AP value without knowing where the baseline sits for that class is like reading a test score without knowing what the passing mark was: a number that looks moderate might be excellent relative to a low baseline, or poor relative to a high one.

The macro AP printed in the title averages the three per-class AP values equally, in the same way that macro AUC did for the ROC curves. `average_precision_score()` accepts the full binarised label matrix and the full score matrix directly and handles the averaging internally when `average='macro'` is passed, so no manual loop is needed for the summary statistic.

Comparing these PR curves to the ROC curves from the previous section will typically reveal that the classes are harder to distinguish here than the ROC plot suggested. The overlapping clusters and label noise in the dataset mean that precision drops off as recall increases, and the curves sit closer to their respective baselines than the ROC curves sat relative to the diagonal. This is the PR curve being honest about a problem the ROC curve was softening.


---

## 9. Evaluating Unsupervised Methods

🔙 [Back to Table of Contents](#table-of-contents)

### 9.1 The Core Problem

Everything we have covered so far assumes that ground-truth labels exist. In supervised learning, evaluation is straightforward: you compare the model's predictions against the correct answers and measure how often it gets things right. Clustering has no equivalent anchor. The algorithm decides both how many groups to form and which examples belong to each one, and there are no correct answers to check against. We have to assess quality using the data itself.

This is a genuinely harder problem than it might first appear. Any clustering algorithm can achieve perfect compactness by assigning every point to its own cluster of size one: every point is trivially the closest thing to itself. It can also achieve a kind of trivial separation by lumping everything into a single cluster, which is maximally distant from nothing. Neither extreme is useful. What we want is a clustering that is compact and well separated at the same time, and **internal evaluation metrics** are designed to measure exactly that balance without needing labels.

The two properties these metrics capture are:

* **Compactness.** Points within the same cluster should be close to one another. A compact cluster has low intra-cluster scatter: its points do not spread far from the cluster centroid. The tighter the cluster, the more confidently we can say that the points it contains are genuinely similar to each other.

* **Separation.** Different clusters should be far apart from one another. A well-separated clustering has large gaps between the centroids or the outer boundaries of each group. If clusters overlap heavily, any boundary we draw between them is essentially arbitrary.

A good clustering maximises both simultaneously. The difficulty is that these two goals are in tension: increasing $k$, the number of clusters, always improves compactness because smaller groups are easier to keep tight, but it can destroy separation by splitting what should be a single natural grouping into fragments that sit close together. The metrics introduced in Sections 9.2 to 9.4 each capture this trade-off in a different way, and understanding their differences helps you choose the right one for a given situation.

Figure 13 makes the contrast between a good and a poor clustering concrete. Both panels use the same cluster centres and the same number of points, but the standard deviation of each cluster differs. The dashed circles mark the RMS radius of each cluster, a direct measure of compactness. In the left panel the circles are small and do not overlap; in the right panel they are large and intersect heavily. Any internal evaluation metric worth using should score these two situations very differently.


In [ ]:
# Listing 15.
%matplotlib widget
from visualisations.Figure_13 import show
show()

### 9.2 WCSS and the Elbow Method

The most direct measure of compactness is the **Within-Cluster Sum of Squares (WCSS)**, also called **inertia**. For each cluster, it measures how tightly packed the points are around their centroid by summing the squared distances from every point to the centroid of its assigned cluster, then totalling across all clusters:

$$\text{WCSS} = \sum_{k=1}^{K} \sum_{\mathbf{x} \in C_k} \|\mathbf{x} - \boldsymbol{\mu}_k\|^{2}$$

where $C_k$ is the set of points assigned to cluster $k$ and $\boldsymbol{\mu}_k$ is the centroid of that cluster. Lower WCSS means tighter, more compact clusters.

There is an important catch. WCSS always decreases as $K$ increases, because adding more clusters allows each one to be smaller and tighter. At the extreme where $K$ equals the number of data points, every point is its own cluster and WCSS is exactly zero. This means you cannot choose $K$ by minimising WCSS directly: any larger value of $K$ will always win. What matters is not the absolute value but the rate at which WCSS improves as $K$ grows.

The **elbow method** exploits this. You fit k-means for a range of $K$ values, record the WCSS at each one, and plot the result. When the data contains genuine natural groupings, WCSS falls steeply up to the true cluster count and then levels off into a plateau of diminishing returns. The point where the curve bends, the elbow, is the natural choice of $K$. Beyond it, adding another cluster buys very little compactness improvement while increasing the complexity of the model unnecessarily.

In scikit-learn, WCSS is computed automatically during fitting and stored as the `inertia_` attribute of a fitted `KMeans` object:

```python
from sklearn.cluster import KMeans

km = KMeans(n_clusters=4, n_init=10, random_state=0)
km.fit(X)

print(km.inertia_)   # WCSS for k=4
```

The `n_init=10` argument runs k-means ten times with different random initialisations and keeps the result with the lowest inertia. This is important because k-means is sensitive to its starting centroids and a single run may converge to a poor local minimum.

To apply the elbow method, fit k-means for a range of $K$ values and collect the inertia at each step:

```python
k_range   = range(1, 11)
wcss_vals = []

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=0)
    km.fit(X)
    wcss_vals.append(km.inertia_)
```

Plotting `wcss_vals` against `k_range` gives the elbow curve. A complementary view is to plot the incremental improvement, the drop in WCSS when going from $K-1$ to $K$ clusters, as a bar chart. The bar heights fall sharply up to the true cluster count and then flatten, making the elbow easier to identify visually than on the raw curve alone.

Figure 14 demonstrates both views on a synthetic dataset generated with four natural clusters. The left panel shows the raw WCSS curve; the right panel shows the incremental improvement at each step. The dashed red line at $K = 4$ marks the true cluster count.

> ⚠️ **Warning:** The elbow method is a heuristic, not a guarantee. On real datasets the elbow is often ambiguous: the curve may bend gradually rather than sharply, or there may be two plausible candidates. When this happens, use the elbow as a starting point and validate your choice of $K$ using the silhouette score or Davies-Bouldin Index, both of which are introduced in the sections that follow.

In [ ]:
# Listing 16.
%matplotlib widget
from visualisations.Figure_14 import show
show()

Before plotting the elbow curve, it is worth looking at the raw WCSS values in a table and applying the elbow method directly to the numbers. The code below generates a four-cluster synthetic dataset, fits k-means for $K = 1$ to $10$, and prints the WCSS and the incremental improvement at each step. It then identifies the elbow programmatically by finding where the improvement drops most sharply, that is, where the second difference of the WCSS sequence is largest, and flags that row in the output.

The improvement column is the drop in WCSS going from $K-1$ to $K$ clusters. A large number means adding that cluster recovered genuine structure in the data; a small number means you are past the natural groupings and are simply splitting existing clusters into smaller fragments. The elbow is the step where a previously large improvement suddenly becomes small, and everything beyond it stays small. That is the value of $K$ the elbow method recommends.

In [ ]:
# Listing 17.
# Generate a synthetic dataset with 4 natural clusters
X, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=3)

# Fit k-means for k = 1 to 10 and record WCSS (inertia) at each step.
# n_init=10 runs k-means 10 times with different random initialisations
# and keeps the result with the lowest inertia, reducing sensitivity to
# the starting centroid positions.
k_range   = range(1, 11)
wcss_vals = []

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=0)
    km.fit(X)
    wcss_vals.append(km.inertia_)

# Apply the elbow method: find the k where the incremental improvement
# drops most sharply. This is the point of diminishing returns.
improvements = [wcss_vals[i - 1] - wcss_vals[i] for i in range(1, len(wcss_vals))]

# The elbow is the k at which the improvement is smallest relative to
# the previous step, i.e. where the rate of gain collapses.
# We find it by looking for the largest drop in improvement (second difference).
second_diff = [improvements[i - 1] - improvements[i] for i in range(1, len(improvements))]
elbow_k     = second_diff.index(max(second_diff)) + 2   # +2 because k starts at 1 and improvements at k=2

# Print the WCSS, improvement, and flag the elbow
print(f'{"k":>4}  {"WCSS":>10}  {"Improvement":>14}  {"":>6}')
print('-' * 42)
for i, (k, wcss) in enumerate(zip(k_range, wcss_vals)):
    improvement = improvements[i - 1] if i > 0 else None
    impr_str    = f'{improvement:>14.1f}' if improvement is not None else f'{"—":>14}'
    marker      = '  <-- elbow' if k == elbow_k else ''
    print(f'{k:>4}  {wcss:>10.1f}  {impr_str}{marker}')

print(f'\nElbow method suggests k = {elbow_k}')


---

### 9.3 Silhouette Score

The silhouette score addresses a limitation of WCSS: it only measures compactness and has no notion of how well the clusters are separated from one another. The silhouette score captures both properties simultaneously, and does so at the level of individual points rather than the clustering as a whole, which makes it both more informative and more interpretable.

For each point $i$, two distances are computed:

$$a(i) = \text{mean distance from } i \text{ to all other points in its own cluster}$$

$$b(i) = \text{mean distance from } i \text{ to all points in the nearest neighbouring cluster}$$

$a(i)$ measures compactness: how tightly packed the point's own cluster is around it. $b(i)$ measures separation: how far the point sits from the closest cluster it does not belong to. The silhouette score for point $i$ combines these into a single value:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i),\, b(i))}$$

The result always lies in $[-1, +1]$:

| Score | Meaning |
|-------|---------|
| $s(i) = +1$ | The point is deep inside its own cluster and far from all others. Ideal. |
| $s(i) \approx 0$ | The point sits on or near the boundary between two clusters. |
| $s(i) = -1$ | The point is likely misassigned: it is closer to a neighbouring cluster than to its own. |

The **mean silhouette score** across all points summarises the overall quality of the clustering. Higher is better, and a value above 0.5 is generally considered a reasonable clustering. A value below 0.2 suggests substantial overlap or a poor choice of $K$.

Scikit-learn provides two functions for computing silhouette scores, both in `sklearn.metrics`:

`silhouette_score(X, labels)` returns the mean silhouette score across all points. This is the single summary number you would use to compare clusterings at different values of $K$.

`silhouette_samples(X, labels)` returns an individual silhouette score for every point in the dataset. This is useful for building a silhouette plot, which shows the score distribution within each cluster and makes it easy to identify which clusters are well-formed and which contain a large number of borderline or misassigned points.

```python
from sklearn.metrics import silhouette_score, silhouette_samples

# Mean silhouette score for the full clustering
score = silhouette_score(X, labels)

# Per-point scores, one value per sample
sample_scores = silhouette_samples(X, labels)
```

Both functions accept any cluster label array, so they work with k-means, DBSCAN, agglomerative clustering, or any other algorithm that produces integer labels.

The minimum working example below fits k-means for $K = 2$ to $8$ on a synthetic four-cluster dataset, computes the mean silhouette score at each $K$, and prints the results. The correct value of $K$ should achieve the highest mean score.

```python
from sklearn.cluster  import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics  import silhouette_score

X, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=3)

# Silhouette score is undefined for k=1 (there is no neighbouring cluster
# to compute b(i) against), so the range starts at k=2.
print(f'{"k":>4}  {"Silhouette Score":>18}')
print('-' * 26)

for k in range(2, 9):
    km     = KMeans(n_clusters=k, n_init=10, random_state=0)
    labels = km.fit_predict(X)

    # silhouette_score() takes the raw data and the cluster labels.
    # It computes pairwise distances internally using Euclidean distance
    # by default (metric='euclidean'). Pass metric='cosine' or any
    # scipy distance string to use a different distance measure.
    score = silhouette_score(X, labels)
    marker = '  <-- best' if k == 4 else ''
    print(f'{k:>4}  {score:>18.4f}{marker}')
```

Here's the code in action.

In [ ]:
# Listing 18.
X, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=3)

# Silhouette score is undefined for k=1 (there is no neighbouring cluster
# to compute b(i) against), so the range starts at k=2.
print(f'{"k":>4}  {"Silhouette Score":>18}')
print('-' * 26)

best_k, best_score = None, -1

for k in range(2, 9):
    km     = KMeans(n_clusters=k, n_init=10, random_state=0)
    labels = km.fit_predict(X)

    # silhouette_score() takes the raw data and the cluster labels.
    # It computes pairwise distances internally using Euclidean distance
    # by default (metric='euclidean'). Pass metric='cosine' or any
    # scipy distance string to use a different distance measure.
    score = silhouette_score(X, labels)

    if score > best_score:
        best_score, best_k = score, k

    print(f'{k:>4}  {score:>18.4f}')

print(f'\nBest k by silhouette score: k = {best_k}  (score = {best_score:.4f})')

---

### 9.4 Davies-Bouldin Index and Dunn Index

#### Davies-Bouldin Index (DBI)

The Davies-Bouldin Index evaluates a clustering by asking, for each cluster, how similar it is to its most similar neighbour. Similarity here means a high ratio of combined scatter to centroid separation: clusters that are spread out and sit close together score poorly, while clusters that are tight and well separated score well.

For each cluster $k$, the index finds the neighbouring cluster $l$ that produces the worst (highest) similarity ratio, then averages these worst-case ratios across all clusters:

$$DBI = \frac{1}{K} \sum_{k=1}^{K} \max_{l \neq k} \frac{s_k + s_l}{d(\boldsymbol{\mu}_k,\, \boldsymbol{\mu}_l)}$$

where $s_k$ is the mean distance from points in cluster $k$ to their centroid (a measure of scatter), and $d(\boldsymbol{\mu}_k, \boldsymbol{\mu}_l)$ is the Euclidean distance between centroids. **Lower DBI is better.** A value of 0 indicates perfectly compact, perfectly separated clusters.

The DBI penalises two things simultaneously: clusters that are internally scattered (large $s_k$) and clusters whose centroids sit close together (small $d$). A clustering that scores well on DBI must be good on both counts, not just one.

Scikit-learn computes the DBI with a single function from `sklearn.metrics`:

```python
from sklearn.metrics import davies_bouldin_score

# Takes the raw data and the cluster label array.
# Lower is better; 0.0 is a perfect clustering.
dbi = davies_bouldin_score(X, labels)
```

The minimum working example below fits k-means for $K = 2$ to $8$ and computes DBI at each step. The value of $K$ with the lowest DBI is the one the index recommends.


In [ ]:
# Listing 19.
X, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=3)

print(f'{"k":>4}  {"Davies-Bouldin Index":>22}')
print('-' * 30)

best_k, best_dbi = None, float('inf')

for k in range(2, 9):
    km     = KMeans(n_clusters=k, n_init=10, random_state=0)
    labels = km.fit_predict(X)

    # davies_bouldin_score() takes the raw data and cluster labels.
    # Lower values indicate better-defined, more separated clusters.
    dbi = davies_bouldin_score(X, labels)

    if dbi < best_dbi:
        best_dbi, best_k = dbi, k

    print(f'{k:>4}  {dbi:>22.4f}')

print(f'\nBest k by Davies-Bouldin Index: k = {best_k}  (DBI = {best_dbi:.4f})')

#### Dunn Index

The Dunn Index takes a complementary approach. Rather than averaging over all clusters, it focuses on the extremes: the smallest gap between any two clusters and the largest diameter of any single cluster. The ratio of these two quantities defines the index:

$$D = \frac{\displaystyle\min_{i \neq j}\, d(C_i, C_j)}{\displaystyle\max_{k}\, \text{diam}(C_k)}$$

**Higher Dunn is better.** A good clustering has clusters that are far apart from one another (large numerator) and individually compact (small denominator). The index is most sensitive to outliers and boundary points because the minimum inter-cluster distance and maximum diameter are both determined by the most extreme point pairs in the dataset.

Scikit-learn does not include a built-in Dunn Index function, so it requires a manual implementation. The key ingredients are `cdist()` from `scipy.spatial.distance`, which computes pairwise distances between sets of points efficiently, and straightforward loop logic to find the minimum inter-cluster gap and maximum cluster diameter.

The full code sample that shows you how to compute this is given below.

> ⚠️ **Warning:** The Dunn Index requires computing all pairwise distances between points within and between clusters. For a dataset of $n$ points this scales as $O(n^2)$, making it impractical for large datasets. On datasets of more than a few thousand points, prefer the silhouette score or Davies-Bouldin Index, both of which are substantially cheaper to compute.

In [ ]:
# Listing 20.
# Create a data set.
X, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=3)

def dunn_index(X, labels):
    """
    Compute the Dunn Index for a clustering result.

    Parameters
    ----------
    X      : (n_samples, n_features) array of data points.
    labels : (n_samples,) integer array of cluster assignments.

    Returns
    -------
    float : Dunn Index. Higher is better.
    """
    unique_labels = np.unique(labels)
    clusters      = [X[labels == k] for k in unique_labels]

    # Minimum inter-cluster distance: smallest distance between any
    # point in cluster i and any point in cluster j, across all pairs.
    min_inter = float('inf')
    for i in range(len(clusters)):
        for j in range(i + 1, len(clusters)):
            # cdist() computes the full pairwise distance matrix between
            # two sets of points. min() gives the closest pair.
            dist = cdist(clusters[i], clusters[j]).min()
            min_inter = min(min_inter, dist)

    # Maximum intra-cluster diameter: largest distance between any two
    # points within the same cluster, across all clusters.
    max_intra = 0.0
    for cluster in clusters:
        if len(cluster) > 1:
            diam      = cdist(cluster, cluster).max()
            max_intra = max(max_intra, diam)

    return min_inter / max_intra if max_intra > 0 else float('inf')


print(f'{"k":>4}  {"Dunn Index":>14}')
print('-' * 22)

best_k, best_dunn = None, -float('inf')

for k in range(2, 9):
    km     = KMeans(n_clusters=k, n_init=10, random_state=0)
    labels = km.fit_predict(X)
    dunn   = dunn_index(X, labels)

    if dunn > best_dunn:
        best_dunn, best_k = dunn, k

    print(f'{k:>4}  {dunn:>14.4f}')

print(f'\nBest k by Dunn Index: k = {best_k}  (Dunn = {best_dunn:.4f})')

---

Figure 15 lets you build direct geometric intuition for the three metrics introduced in this section. The two sliders give independent control over the two properties every internal metric is trying to measure: how spread apart the clusters are from one another, and how compact each cluster is internally. Dragging the spread slider leftward pushes the three centroids together until the clusters overlap; dragging it rightward pulls them apart into clean separation. The compactness slider controls how tightly points cluster around their centroid: high compactness means a tight, dense group; low compactness means a loose, diffuse cloud.

Watch how the three metrics respond differently to each change. The Dunn Index is the most sensitive to the spread slider because its numerator is the minimum inter-cluster distance, which collapses quickly as clusters begin to touch. The Davies-Bouldin Index responds strongly to both sliders because it penalises both internal scatter and centroid proximity simultaneously. The silhouette score is perhaps the most balanced: it degrades smoothly as either property worsens, and its per-point interpretation (how much closer each point is to its own cluster than to its nearest neighbour) maps directly onto what you can see happening in the scatter plot.

The table below summarises the three metrics covered in this section:

| Metric | Measures | Direction | Scikit-learn function | Cost |
|---|---|---|---|---|
| Silhouette Score | Compactness and separation per point | Higher is better (+1 max) | `silhouette_score()` | Medium |
| Davies-Bouldin Index | Average worst-case cluster similarity | Lower is better (0 min) | `davies_bouldin_score()` | Low |
| Dunn Index | Ratio of min inter-cluster gap to max diameter | Higher is better (0 min) | Manual (scipy `cdist`) | High |

No single metric is universally best. The silhouette score is the most interpretable and works well as a general-purpose choice. The Davies-Bouldin Index is fast and reliable for comparing clusterings at different values of $K$. The Dunn Index gives the most direct geometric answer but becomes impractical on large datasets due to its $O(n^2)$ distance computation. In practice, computing two or three of these together and looking for agreement is more robust than relying on any one alone.

In [ ]:
# Listing 21.
%matplotlib widget
from visualisations.Figure_15 import show
show()



---

## 10. Summary

### Supervised Evaluation

| Tool | What it measures | Best used when |
|------|-----------------|----------------|
| **ROC curve** | TPR vs FPR across all decision thresholds | Classes are roughly balanced; comparing classifiers by ranking |
| **AUC** | Probability that the model ranks a positive above a negative | Single-number ranking comparison; threshold-free |
| **PR curve** | Precision vs Recall across all decision thresholds | Imbalanced classes; performance on the minority class matters |
| **Average Precision (AP)** | Area under the PR curve | Summary statistic for PR performance |

### Unsupervised Evaluation

| Metric | Formula | Direction | What it measures |
|--------|---------|-----------|-----------------|
| **WCSS (inertia)** | $\sum_k \sum_{\mathbf{x} \in C_k} \|\mathbf{x} - \boldsymbol{\mu}_k\|^2$ | Lower = better | Compactness only |
| **Silhouette** | $s(i) = (b(i) - a(i)) / \max(a(i), b(i))$ per point | Higher = better (+1 ideal) | Compactness + Separation |
| **Davies-Bouldin** | Mean worst-case scatter-to-separation ratio | Lower = better (0 ideal) | Compactness + Separation |
| **Dunn Index** | Min inter-cluster distance / Max intra-cluster diameter | Higher = better | Separation + Compactness |

### Key Rules of Thumb

- Use **PR curves** (not ROC) on imbalanced datasets, since the PR curve gives an honest picture of performance on the minority class.
- Use **multiple clustering metrics together**: agreement across WCSS (elbow), silhouette, and DBI gives confidence in your $k$ selection.
- When ROC curves **cross**, zoom into your operating region: global AUC can favour the wrong model at the threshold you actually need.
- When clustering metrics **disagree**, inspect the data visually, apply domain knowledge, or consider that the cluster structure may genuinely be ambiguous.

---


## 11. References

Davis, J. and Goadrich, M. (2006) 'The relationship between Precision-Recall and ROC curves', in *Proceedings of the 23rd International Conference on Machine Learning (ICML)*. New York: ACM, pp. 233–240.

Davies, D.L. and Bouldin, D.W. (1979) 'A cluster separation measure', *IEEE Transactions on Pattern Analysis and Machine Intelligence*, 1(2), pp. 224–227.

Dunn, J.C. (1974) 'Well-separated clusters and optimal fuzzy partitions', *Journal of Cybernetics*, 4(1), pp. 95–104.

Fawcett, T. (2006) 'An introduction to ROC analysis', *Pattern Recognition Letters*, 27(8), pp. 861–874.

Pedregosa, F. *et al.* (2011) 'Scikit-learn: Machine learning in Python', *Journal of Machine Learning Research*, 12, pp. 2825–2830.

Rousseeuw, P.J. (1987) 'Silhouettes: a graphical aid to the interpretation and validation of cluster analysis', *Journal of Computational and Applied Mathematics*, 20, pp. 53–65.
